# Kasus 07 · Irigasi pintar IoT — Tahap 1: Pencarian Model (versi koreksi)

Tujuan: membandingkan beberapa model untuk memprediksi `perlu_siram` (perlu disiram sekarang atau tidak)
per petak per jam, lalu membandingkannya dengan **baseline wajib** (siram kalau `kelembaban_tanah_pct` < 35).

- **Metrik utama:** F1 untuk kelas `perlu_siram = 1` (kelas positif cuma ~4%).
- **Split berdasarkan waktu:** train Agustus–September, validation 1–14 Oktober, test 15–31 Oktober.
- **Test set belum dibuka** di notebook ini.

### Apa yang dikoreksi dari versi sebelumnya
1. **Kebocoran data (leakage).** Di jam penyiraman, sensor kelembaban tanah sudah mencatat kondisi *setelah*
   disiram. Versi lama memakai pembacaan jam itu dan fitur `delta_tanah`, jadi model belajar
   "tanah tiba-tiba basah = tadi disiram". Sekarang model hanya memakai **pembacaan terakhir sebelum keputusan**
   (jam sebelumnya), sama seperti kondisi saat app dipakai.
2. **Pembersihan lebih lengkap:** sensor macet, kelembaban 0%, cahaya negatif, label positif di malam hari.
3. **Batas aman per tanaman diambil dari data train**, bukan tebakan.
4. **Cek stabilitas** peringkat model di beberapa periode waktu, bukan cuma satu validation.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
RANDOM_STATE = 42

## 1. Load data

In [ ]:
kandidat = [Path("irigasi.csv"), Path("data/irigasi.csv"), Path("../data/irigasi.csv")]
DATA_PATH = next((p for p in kandidat if p.exists()), None)
assert DATA_PATH is not None, "irigasi.csv tidak ditemukan. Taruh di folder yang sama dengan notebook atau di data/."

df_raw = pd.read_csv(DATA_PATH, parse_dates=["waktu"])
print("Ukuran data:", df_raw.shape)
print("Proporsi target:", df_raw["perlu_siram"].value_counts(normalize=True).round(4).to_dict())
df_raw.head()

## 2. Investigasi masalah data

### 2a. Masalah umum: duplikat, nama tanaman, nilai kosong, nilai mustahil

In [ ]:
print("Duplikat (petak + waktu):", df_raw.duplicated(subset=["waktu", "petak"]).sum())
print("\nPenulisan nama tanaman:\n", df_raw["tanaman"].value_counts().to_dict())
print("\nNilai kosong:\n", df_raw.isna().sum()[lambda s: s > 0].to_dict())
print("\nRentang nilai:")
df_raw[["kelembaban_tanah_pct", "suhu_udara_C", "kelembaban_udara_pct", "cahaya_lux",
        "curah_hujan_mm", "prakiraan_hujan_6jam_mm"]].agg(["min", "max"]).T

Temuan: kelembaban tanah ada yang **0%** dan **di atas 100%**, cahaya ada yang **negatif**. Semuanya mustahil.

Setiap petak selalu berisi tanaman dan jenis tanah yang sama:

In [ ]:
df = df_raw.drop_duplicates(subset=["waktu", "petak"]).sort_values(["petak", "waktu"]).reset_index(drop=True)
df["tanaman"] = df["tanaman"].str.strip().str.lower().replace({"cabe": "cabai", "lettuce": "selada"})
df.groupby("petak")[["tanaman", "jenis_tanah"]].first()

### 2b. Sensor macet

Sensor murah bisa macet: nilainya tidak berubah berjam-jam. Kita hitung panjang "run" nilai yang persis sama
berturut-turut. Run 4 jam atau lebih dianggap macet.

In [ ]:
KOL_TANAH = "kelembaban_tanah_pct"
sama_dgn_sebelumnya = df[KOL_TANAH].eq(df.groupby("petak")[KOL_TANAH].shift(1))
id_run = (~sama_dgn_sebelumnya).cumsum()
panjang_run = df.groupby(["petak", id_run])[KOL_TANAH].transform("size")
macet = panjang_run >= 4

print("Baris yang sensornya macet:", macet.sum())
print(df[macet].groupby("petak").size().to_dict())

contoh = df[df["petak"] == "P05"].set_index("waktu")[KOL_TANAH]
awal = df.loc[macet & (df["petak"] == "P05"), "waktu"].min()
contoh[awal - pd.Timedelta(hours=12): awal + pd.Timedelta(hours=60)].plot(figsize=(9, 3), marker=".")
plt.title("Contoh sensor macet di P05 (garis datar)")
plt.ylabel("kelembaban tanah (%)")
plt.show()

### 2c. Kebocoran data (leakage): sensor di jam penyiraman sudah mencatat tanah basah

Lihat contoh P01 tanggal 13 Agustus. Jam 05–08 tanah kering dan labelnya 1. Di jam 09 labelnya masih 1,
tapi sensor sudah membaca 53%, karena pembacaan itu dicatat **setelah** petak disiram.

In [ ]:
df[(df["petak"] == "P01") & df["waktu"].between("2025-08-13 04:00", "2025-08-13 11:00")][
    ["waktu", "tanaman", KOL_TANAH, "perlu_siram"]]

In [ ]:
df["_delta"] = df.groupby("petak")[KOL_TANAH].diff()
print("Perubahan kelembaban tanah dibanding jam sebelumnya, per label:")
print(df.groupby("perlu_siram")["_delta"].describe().round(2))
pos = df[df["perlu_siram"] == 1]
print(f"\nBaris positif yang kelembabannya melonjak > 5 poin di jam itu juga: "
      f"{(pos['_delta'] > 5).sum()} dari {len(pos)}")
df = df.drop(columns="_delta")

**Kesimpulan:** sekitar seperempat baris positif sudah "melihat" hasil penyiraman. Kalau model memakai pembacaan
jam itu, skornya bagus tapi palsu, karena di dunia nyata keputusan diambil **sebelum** menyiram.

**Perbaikan:** fitur kelembaban tanah yang dipakai adalah **pembacaan terakhir sebelum jam itu** (`tanah_terakhir`
= nilai jam sebelumnya). Ini sama dengan kondisi di app: pengelola memasukkan pembacaan sensor terbaru, lalu
model memutuskan perlu disiram atau tidak.

### 2d. Label positif di malam hari

Brief bilang agronom cuma menyiram siang hari. Kita lihat proporsi label positif per jam:

In [ ]:
df["jam"] = df["waktu"].dt.hour
per_jam = df.groupby("jam")["perlu_siram"].mean()
plt.figure(figsize=(10, 3))
plt.bar(per_jam.index, per_jam.values)
plt.xticks(range(24))
plt.xlabel("jam")
plt.title("Proporsi perlu_siram per jam")
plt.ylabel("proporsi positif")
plt.show()

malam_pos = (~df["jam"].between(5, 17)) & (df["perlu_siram"] == 1)
print("Label positif di luar jam 05–17:", malam_pos.sum())

Siang hari di data ini = jam **05–17**. Label positif di luar jam itu (40 baris) bertentangan dengan kebiasaan
agronom, jadi dianggap **label salah** dan diubah jadi 0, **hanya di data train**. Label validation/test tidak
diubah supaya evaluasinya tetap jujur.

## 3. Pembersihan

| Masalah | Cara membereskan |
|---|---|
| Duplikat petak + waktu | dibuang (sudah di atas) |
| Nama tanaman tidak konsisten | disamakan (sudah di atas) |
| Kelembaban tanah 0% atau > 100% | NaN, lalu interpolasi per petak |
| Sensor kelembaban macet (≥ 4 jam nilai sama) | NaN, lalu interpolasi per petak |
| Cahaya negatif | NaN, lalu interpolasi per petak |
| Suhu kosong | interpolasi per petak |
| Prakiraan hujan kosong | diisi 0 |
| Label positif di malam hari | diubah jadi 0 (hanya train) |

In [ ]:
mustahil_tanah = ~df[KOL_TANAH].between(0.1, 100)
cahaya_negatif = df["cahaya_lux"] < 0
print(f"Kelembaban tanah mustahil: {mustahil_tanah.sum()} | sensor macet: {macet.sum()} | "
      f"cahaya negatif: {cahaya_negatif.sum()}")

df.loc[mustahil_tanah | macet, KOL_TANAH] = np.nan
df.loc[cahaya_negatif, "cahaya_lux"] = np.nan
for kol in [KOL_TANAH, "suhu_udara_C", "cahaya_lux"]:
    df[kol] = df.groupby("petak")[kol].transform(lambda s: s.interpolate(limit_direction="both"))
df["prakiraan_hujan_6jam_mm"] = df["prakiraan_hujan_6jam_mm"].fillna(0)

print("Sisa nilai kosong:", int(df.isna().sum().sum()))

## 4. Fitur

- `tanah_terakhir`: pembacaan kelembaban tanah jam sebelumnya (perbaikan leakage).
- `jam` dan `siang` (05–17): agronom cuma menyiram siang hari.
- `selisih_batas`: `tanah_terakhir` dikurangi batas aman tanaman. Batasnya dihitung dari **data train saja**,
  yaitu titik tengah antara kelembaban yang biasanya disiram dan yang biasanya tidak disiram (siang hari).
- One-hot `tanaman` dan `jenis_tanah`.

In [ ]:
df["siang"] = df["jam"].between(5, 17).astype(int)
df["tanah_terakhir"] = df.groupby("petak")[KOL_TANAH].shift(1)
df = df.dropna(subset=["tanah_terakhir"]).reset_index(drop=True)  # jam pertama tiap petak tidak punya jam sebelumnya

m_train = df["waktu"] < "2025-10-01"
m_val   = (df["waktu"] >= "2025-10-01") & (df["waktu"] < "2025-10-15")
m_test  = df["waktu"] >= "2025-10-15"   # JANGAN dipakai sampai evaluasi akhir

siang_train = df[m_train & (df["siang"] == 1)]
ringkas_batas = siang_train.groupby(["tanaman", "perlu_siram"])["tanah_terakhir"].quantile([0.05, 0.5, 0.9]).unstack()
display(ringkas_batas.round(1))

BATAS_AMAN = {}
for t, s in siang_train.groupby("tanaman"):
    q90_siram = s.loc[s["perlu_siram"] == 1, "tanah_terakhir"].quantile(0.9)
    q05_tidak = s.loc[s["perlu_siram"] == 0, "tanah_terakhir"].quantile(0.05)
    BATAS_AMAN[t] = round((q90_siram + q05_tidak) / 2)
print("Batas aman per tanaman (dari data train):", BATAS_AMAN)

Hasilnya sesuai pengetahuan dari brief: **selada paling haus** (batas paling tinggi), **cabai paling tahan kering**.
Ini juga menjelaskan kenapa baseline `< 35` buruk: batasnya terlalu tinggi untuk cabai dan terlalu rendah untuk selada.

In [ ]:
df["selisih_batas"] = df["tanah_terakhir"] - df["tanaman"].map(BATAS_AMAN)

FITUR_NUM = ["umur_tanaman_hari", "tanah_terakhir", "suhu_udara_C", "kelembaban_udara_pct", "cahaya_lux",
             "curah_hujan_mm", "prakiraan_hujan_6jam_mm", "jam", "siang", "selisih_batas"]
X = pd.concat([df[FITUR_NUM], pd.get_dummies(df[["tanaman", "jenis_tanah"]]).astype(int)], axis=1)
y = df["perlu_siram"]

# Label untuk training: positif di malam hari dianggap salah label
y_latih = y.copy()
y_latih[(df["siang"] == 0) & (y_latih == 1)] = 0

X_train, y_train = X[m_train], y_latih[m_train]
X_val,   y_val   = X[m_val],   y[m_val]

for nama, m in [("train", m_train), ("validation", m_val), ("test (disisihkan)", m_test)]:
    print(f"{nama:18s}: {m.sum():6d} baris, proporsi positif = {y[m].mean():.3f}")
print("Jumlah fitur:", X.shape[1])

## 5. Baseline

1. **Baseline wajib dari brief:** siram kalau `kelembaban_tanah_pct` < 35 (dipakai persis seperti di brief).
2. **Aturan domain (tanpa ML):** siram kalau tanah di bawah batas aman tanamannya **dan** siang hari **dan**
   prakiraan hujan 6 jam ≤ 2 mm. Ini menguji apakah ML benar-benar lebih baik daripada aturan yang disusun
   dari pengetahuan agronom.

In [ ]:
def ringkas(y_true, y_pred):
    return {"F1": f1_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred)}

val = df[m_val]
# baseline wajib memakai kelembaban tanah mentah, persis seperti aturan di brief
mentah = df_raw.drop_duplicates(subset=["waktu", "petak"]).set_index(["waktu", "petak"])[KOL_TANAH]
tanah_mentah_val = mentah.reindex(pd.MultiIndex.from_frame(val[["waktu", "petak"]])).values
pred_wajib = (tanah_mentah_val < 35).astype(int)
pred_domain = ((val["selisih_batas"] < 0) & (val["siang"] == 1)
               & (val["prakiraan_hujan_6jam_mm"] <= 2)).astype(int)

hasil_wajib, hasil_domain = ringkas(y_val, pred_wajib), ringkas(y_val, pred_domain)
print("Baseline wajib (< 35):", {k: round(v, 3) for k, v in hasil_wajib.items()})
print("Aturan domain        :", {k: round(v, 3) for k, v in hasil_domain.items()})

## 6. Membandingkan model di validation

Semua model memakai bobot kelas karena kelas positif sangat sedikit. Dilaporkan F1 di threshold 0,5 dan
di threshold terbaik.

⚠️ F1 di threshold terbaik **terlalu optimis** karena threshold dicari di validation yang sama.
Dipakai hanya untuk melihat potensi model.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
    "Decision Tree (depth 6)": DecisionTreeClassifier(
        max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=3, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=3, class_weight="balanced",
        n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, class_weight="balanced", random_state=RANDOM_STATE),
}

def threshold_terbaik(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    i = np.argmax(f1[:-1])
    return thr[i], f1[i]

baris = [
    {"Model": "Baseline wajib (< 35)", "F1 @0.5": hasil_wajib["F1"],
     "Precision @0.5": hasil_wajib["Precision"], "Recall @0.5": hasil_wajib["Recall"]},
    {"Model": "Aturan domain", "F1 @0.5": hasil_domain["F1"],
     "Precision @0.5": hasil_domain["Precision"], "Recall @0.5": hasil_domain["Recall"]},
]
proba_val = {}
for nama, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_val)[:, 1]
    proba_val[nama] = p
    r = ringkas(y_val, (p >= 0.5).astype(int))
    thr, f1_best = threshold_terbaik(y_val, p)
    baris.append({"Model": nama, "F1 @0.5": r["F1"], "Precision @0.5": r["Precision"],
                  "Recall @0.5": r["Recall"], "Threshold terbaik": thr, "F1 @terbaik": f1_best})

hasil = pd.DataFrame(baris).set_index("Model").round(3).sort_values("F1 @0.5", ascending=False)
hasil

## 7. Cek stabilitas: apakah peringkatnya konsisten di periode lain?

Validation cuma dua minggu (~80 baris positif), jadi selisih 0,01 bisa saja kebetulan. Kita ulangi
pelatihan dengan beberapa titik potong waktu: latih di semua data sebelum periode, uji di periode itu.

In [ ]:
periode = [("2025-08-25", "2025-09-08"), ("2025-09-08", "2025-09-22"),
           ("2025-09-22", "2025-10-01"), ("2025-10-01", "2025-10-15")]

stabil = {}
for nama, model in models.items():
    skor = []
    for awal, akhir in periode:
        m_tr = df["waktu"] < awal
        m_uji = (df["waktu"] >= awal) & (df["waktu"] < akhir)
        mdl = clone(model).fit(X[m_tr], y_latih[m_tr])
        skor.append(f1_score(y[m_uji], mdl.predict_proba(X[m_uji])[:, 1] >= 0.5))
    stabil[nama] = skor

tabel_stabil = pd.DataFrame(stabil, index=[f"{a} s/d {b}" for a, b in periode]).T
tabel_stabil["rata-rata"] = tabel_stabil.mean(axis=1)
tabel_stabil.round(3).sort_values("rata-rata", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for nama, p in proba_val.items():
    prec, rec, _ = precision_recall_curve(y_val, p)
    ax.plot(rec, prec, label=nama)
ax.scatter(hasil_wajib["Recall"], hasil_wajib["Precision"], color="black", zorder=5, label="Baseline wajib (< 35)")
ax.scatter(hasil_domain["Recall"], hasil_domain["Precision"], color="red", marker="x", s=80, zorder=5,
           label="Aturan domain")
ax.set_xlabel("Recall (petak yang perlu disiram berhasil ditangkap)")
ax.set_ylabel("Precision (notifikasi yang benar)")
ax.set_title("Precision–Recall di validation (1–14 Okt)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## 8. Melihat model terbaik lebih dekat: HistGradientBoosting

In [ ]:
hgb = models["HistGradientBoosting"]
pred_hgb = (proba_val["HistGradientBoosting"] >= 0.5).astype(int)

imp = permutation_importance(hgb, X_val, y_val, scoring="f1", n_repeats=10, random_state=RANDOM_STATE)
imp = pd.Series(imp.importances_mean, index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_val, pred_hgb),
                       display_labels=["tidak siram", "siram"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("HistGradientBoosting @0.5 — validation")
imp.tail(10).plot.barh(ax=axes[1])
axes[1].set_title("Permutation importance (turunnya F1 kalau fitur diacak)")
plt.tight_layout()
plt.show()

## 9. Kesimpulan

- **Baseline wajib `< 35` jelek (F1 ≈ 0,19)** karena satu batas tidak cocok untuk semua tanaman, dan tidak
  memperhatikan jam maupun prakiraan hujan.
- **Aturan domain tanpa ML sudah kuat (F1 ≈ 0,80).** Ini pembanding yang jujur: ML harus mengalahkan aturan ini,
  bukan cuma baseline wajib.
- **HistGradientBoosting dan Random Forest jadi model terbaik**, dengan F1 sekitar 0,85 di validation.
  HistGradientBoosting unggul tipis tapi **konsisten di keempat periode** (rata-rata ≈ 0,90 vs 0,89),
  jadi dipilih sebagai model utama. Random Forest jadi cadangan.
- Logistic Regression jelek di threshold 0,5 karena aturan penyiraman berbentuk "dan/atau" (kering **dan** siang
  **dan** tidak mau hujan), dan model linear sulit menangkapnya.
- Fitur terpenting masuk akal: selisih terhadap batas aman tanaman, siang/jam, dan prakiraan hujan.

**Langkah berikutnya:** tuning HistGradientBoosting, pilih threshold di validation berdasarkan trade-off
(telat nyiram vs nyiram percuma), latih ulang, buka test set sekali, lalu simpan `model.joblib` + `metadata.json`.